# 01 · Benchmark Dataset Preparation — FPClose

> **Bản trình bày sạch để đưa vào GitHub — 22/11/2026.**  
> **Vai trò:** kiểm toán và tái lập dữ liệu benchmark, **không phải** triển khai thuật toán FPClose của nhóm.

## Mục tiêu

1. Xác định **9 transaction datasets** đã chuẩn hóa theo định dạng SPMF.
2. Kiểm tra metadata, số giao dịch, thống kê và SHA-256 từ các hồ sơ đã lưu.
3. Giải thích quy trình tạo dữ liệu dẫn xuất: 5 Retail subsets và 6 synthetic datasets.
4. Cho phép kiểm chứng/tái tạo theo yêu cầu **mà không vô tình sửa dữ liệu chuẩn hoặc tốn quota Colab**.

**Nguồn dữ liệu công bố:** [Hugging Face — hytrang/fpclose-fim-benchmarks](https://huggingface.co/datasets/hytrang/fpclose-fim-benchmarks)  
**Drive dự án:** [FIM_Project](https://drive.google.com/drive/folders/1clzRNAL5cHWU5D1GEqFOk18k18_1GD99)  
**Notebook lịch sử gốc:** [01_Data_Preparation.ipynb](https://colab.research.google.com/drive/17kcDkgBy61yCCT83Lg_7dFDMwwBZRBKt)  
**Đề bài:** [Frequent Itemset Mining — HCMUS](https://hcmus-cs-ta.github.io/lab-book/data-mining/projects/itemset/)

### Quy tắc chạy

- **Chạy các phần 1–4** để xem báo cáo kiểm tra nhanh từ file CSV/Markdown có sẵn.
- Các phần cần đọc toàn bộ dataset, tải từ Hugging Face hoặc sinh lại file có cờ `False` theo mặc định.
- **Không có cell nào tự ghi đè** dữ liệu benchmark trên Drive.
- Các phép kiểm tra đầy đủ được ghi nhận trước đó: **9/9 file công bố có SHA-256 khớp**. Notebook này không giả định rằng bạn vừa chạy lại chúng.
- Phải giữ bản đã chuẩn hóa cố định cho cả SPMF và thuật toán nhóm; không chuẩn hóa lại trên bản public.

## 1 · Kết nối môi trường và đường dẫn

Chỉ cần mount Google Drive **một lần**. Ở máy cục bộ, có thể đặt biến môi trường `FIM_PROJECT_ROOT`.

In [ ]:
from pathlib import Path
import os
import hashlib
import csv
import gzip
import random
import re
import tempfile
import pandas as pd

if 'COLAB_GPU' in os.environ or Path('/content').exists():
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/FIM_Project')
else:
    ROOT = Path(os.environ.get('FIM_PROJECT_ROOT', './FIM_Project')).expanduser()

DATA = ROOT / 'data' / 'benchmark'
RAW = ROOT / 'data' / 'raw'
RESULTS = ROOT / 'results'
SCALABILITY = ROOT / 'data' / 'scalability'
SYNTHETIC = ROOT / 'data' / 'synthetic'
HF_REPO = 'hytrang/fpclose-fim-benchmarks'
DATASETS = ['chess', 'mushrooms', 'retail', 'T10I4D100K', 'accidents',
            'connect', 'kosarak', 'pumsb_star', 'T40I10D100K']

required = [RESULTS / 'dataset_statistics.csv',
            RESULTS / 'benchmark_inventory.csv',
            DATA / 'MANIFEST.md']
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Thiếu hồ sơ kiểm tra trên Drive:\n' + '\n'.join(missing))
print('Project root:', ROOT)
print('Datasets expected:', len(DATASETS))
print('Hugging Face:', f'https://huggingface.co/datasets/{HF_REPO}')

## 2 · Kiểm tra hồ sơ thống kê và inventory (nhẹ)

`dataset_statistics.csv` ghi thống kê đo từ dữ liệu sạch; `benchmark_inventory.csv` ghi trạng thái, số giao dịch và hash. Ở bước này **chỉ đọc CSV**, chưa mở toàn bộ dataset. Mọi kết quả PASS bên dưới là *đối chiếu hồ sơ hiện có*, không phải phép kiểm hash mới.

In [ ]:
stats = pd.read_csv(RESULTS / 'dataset_statistics.csv')
inventory = pd.read_csv(RESULTS / 'benchmark_inventory.csv')

assert len(stats) == 9 and len(inventory) == 9
assert set(stats['dataset']) == set(inventory['dataset']) == set(DATASETS)
assert inventory['dataset'].is_unique and stats['dataset'].is_unique

merged = stats.merge(inventory[['dataset', 'transactions', 'sha256', 'status']],
                     on='dataset', validate='one_to_one')
merged['count_record_matches'] = merged['n_transactions'] == merged['transactions']
merged['hash_record_matches'] = (
    merged['clean_sha256'].astype(str).str.lower()
    == merged['sha256'].astype(str).str.lower()
)
assert merged['count_record_matches'].all(), 'Số giao dịch giữa các CSV khác nhau'
assert merged['hash_record_matches'].all(), 'Hash ghi trong các CSV khác nhau'
assert (merged['status'] == 'PASS').all(), 'Có dataset chưa PASS theo inventory'

display(merged[['dataset','n_transactions','n_unique_items',
                'avg_transaction_length','max_transaction_length','density_pct',
                'count_record_matches','hash_record_matches']].sort_values('dataset'))
print('PASS: metadata CSV khớp nhau cho 9/9 dataset.')

## 3 · Đọc MANIFEST và kiểm tra sự nhất quán tài liệu

So SHA-256 theo đúng **khối tên dataset** trong MANIFEST, không chỉ tìm xem chuỗi hash có xuất hiện ở đâu đó. Phần công bố public đã được xác minh ngày 22/11/2026; nếu MANIFEST còn `TODO` thì cần cập nhật tài liệu trước khi commit. Cell này không sửa MANIFEST.

In [ ]:
manifest = (DATA / 'MANIFEST.md').read_text(encoding='utf-8')
manifest_blocks = {}
for name in DATASETS:
    m = re.search(r'^###\s+' + re.escape(name) + r'\s*\n(.*?)(?=^###\s+|^##\s+|\Z)',
                  manifest, flags=re.MULTILINE | re.DOTALL)
    if m is None:
        raise AssertionError(f'MANIFEST thiếu mục: {name}')
    digest_match = re.search(r'Normalized SHA-256:\s*`([a-f0-9]{64})`', m.group(1))
    if digest_match is None:
        raise AssertionError(f'MANIFEST thiếu normalized SHA-256: {name}')
    manifest_blocks[name] = digest_match.group(1)

for row in merged.itertuples(index=False):
    assert manifest_blocks[row.dataset] == str(row.sha256).lower(), row.dataset

print('PASS: 9/9 normalized SHA-256 trong MANIFEST khớp inventory.')
public_url = f'https://huggingface.co/datasets/{HF_REPO}'
if public_url not in manifest or 'TODO after publishing' in manifest:
    print('ACTION REQUIRED: cập nhật link public/TODO trong MANIFEST trước khi commit.')
else:
    print('MANIFEST có link công bố công khai.')

unlogged = stats.loc[stats['id_offset'].astype(str).eq('unknown_original_not_logged'), 'dataset'].tolist()
print('Chưa ghi rõ ID offset lịch sử:', unlogged or 'không có')

## 4 · Kiểm tra tình trạng dữ liệu dẫn xuất (nhẹ)

- **Scalability:** `retail_p010`, `p025`, `p050`, `p075`, `p100`, seed = 2026; lấy prefix của **cùng một hoán vị giao dịch** → lồng nhau.
- **Transaction-length:** 6 file `len05` đến `len30`, mỗi file 10.000 giao dịch, 500 item, tạo mẫu item đồng xuất hiện; seed = 2026.

Cell này kiểm tra file và thống kê đã lưu; **chưa** chứng minh tái tạo từng byte hoặc nestedness. Chức năng tái tạo nằm ở phần tùy chọn sau.

In [ ]:
retail_names = [f'retail_p{p:03d}.txt' for p in (10,25,50,75,100)]
synthetic_names = [f'len{n:02d}.txt' for n in (5,10,15,20,25,30)]
retail_files_ok = all((SCALABILITY / n).is_file() for n in retail_names)
synthetic_files_ok = all((SYNTHETIC / n).is_file() for n in synthetic_names)
print('Retail subsets:', '5/5 present' if retail_files_ok else 'MISSING FILES')
print('Synthetic datasets:', '6/6 present' if synthetic_files_ok else 'MISSING FILES')

summary_path = SYNTHETIC / 'synthetic_statistics.csv'
if summary_path.is_file():
    synthetic_stats = pd.read_csv(summary_path)
    assert len(synthetic_stats) == 6
    display(synthetic_stats)
else:
    print('WARNING: thiếu synthetic_statistics.csv')

## 5 · Tùy chọn: kiểm tra lại SHA-256 của 9 file trên Drive

**Mặc định tắt** để không đọc lại hơn 100 MB dữ liệu. Bật `VERIFY_LOCAL_HASHES = True` nếu cần bằng chứng audit trong phiên mới. Không ghi/đổi bất kỳ file nào.

In [ ]:
VERIFY_LOCAL_HASHES = False

def sha256_file(path, chunk_size=1024*1024):
    hasher = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

if VERIFY_LOCAL_HASHES:
    checked = []
    for row in merged.itertuples(index=False):
        file_path = DATA / f'{row.dataset}.txt'
        actual = sha256_file(file_path)
        checked.append((row.dataset, actual == str(row.sha256).lower()))
        print(row.dataset, 'PASS' if checked[-1][1] else 'MISMATCH')
    assert all(ok for _,ok in checked), 'Dataset bị thay đổi'
    print('LOCAL HASH VERIFICATION: 9/9 PASS')
else:
    print('SKIP: không đọc lại toàn bộ file; dùng hồ sơ SHA-256 đã lưu.')

## 6 · Tùy chọn: kiểm tra bản công khai trên Hugging Face

Hồ sơ đã ghi nhận **9/9 public SHA-256 PASS** ngày 22/11/2026. Chỉ bật khi cần audit lại bản public hoặc sau khi upload phiên bản mới. Đây là bước sử dụng mạng; **không yêu cầu HF_TOKEN** để đọc repo public. Nếu thiếu thư viện, cài `huggingface_hub` trước.

In [ ]:
VERIFY_PUBLIC_HASHES = False

if VERIFY_PUBLIC_HASHES:
    from huggingface_hub import HfApi, hf_hub_download
    all_files = HfApi().list_repo_files(repo_id=HF_REPO, repo_type='dataset', token=False)
    assert 'README.md' in all_files and 'MANIFEST.md' in all_files
    for row in merged.itertuples(index=False):
        name = f'{row.dataset}.txt'
        candidates = [p for p in all_files if Path(p).name == name]
        if len(candidates) != 1:
            raise ValueError(f'Thiếu/trùng file public: {name} -> {candidates}')
        path = hf_hub_download(repo_id=HF_REPO, repo_type='dataset',
                               filename=candidates[0], token=False)
        assert sha256_file(path) == str(row.sha256).lower(), name
        print('[PASS]', name)
    print('PUBLIC HASH VERIFICATION: 9/9 PASS')
else:
    print('SKIP: Hugging Face audit đã được lưu từ phiên trước (9/9 PASS).')

## 7 · Tùy chọn: tái tạo 9 normalized datasets từ raw

**Chỉ chạy khi audit yêu cầu.** Không tải lại nguồn, không sửa dữ liệu public và không ghi đè Drive; ghi file thử vào `/content/fim_reproduction_check/benchmark`. Đối với **5 dataset đầu**, MANIFEST chưa ghi rõ lịch sử item-ID offset, nên thử ứng viên 0/1 và chọn ứng viên **nào cho hash đúng** (nếu có); không tự tuyên bố provenance hoàn chỉnh chỉ nhờ một kết quả khớp.

Nếu cần tái lập từ Internet nguyên gốc, dùng URL và raw SHA-256 trong `MANIFEST.md`; kiểm tra raw trước khi chuẩn hóa.

In [ ]:
REPRODUCE_BENCHMARK_FROM_RAW = False

KNOWN_OFFSETS = {'connect':0, 'kosarak':0, 'pumsb_star':1, 'T40I10D100K':1}

def normalize_raw_spmf(src, dst, offset=0):
    opener = gzip.open if Path(src).suffix == '.gz' else open
    count = 0
    with opener(src, 'rt', encoding='utf-8-sig', newline=None) as fi, \
         Path(dst).open('w', encoding='utf-8', newline='\n') as fo:
        for line_number, raw_line in enumerate(fi, 1):
            line = raw_line.strip()
            if line.startswith(('#', '@', '%')):
                continue
            if not line:
                raise ValueError(f'Blank transaction {src}:{line_number}')
            values = [int(v) for v in line.split()]
            if min(values) + offset <= 0:
                raise ValueError(f'Invalid item ID {src}:{line_number}')
            items = sorted({v + offset for v in values})
            fo.write(' '.join(map(str, items)) + '\n')
            count += 1
    return count

if REPRODUCE_BENCHMARK_FROM_RAW:
    target = Path('/content/fim_reproduction_check/benchmark')
    target.mkdir(parents=True, exist_ok=True)
    for row in stats.itertuples(index=False):
        source = RAW / row.raw_filename
        if not source.is_file():
            raise FileNotFoundError(source)
        offsets = [KNOWN_OFFSETS[row.dataset]] if row.dataset in KNOWN_OFFSETS else [0, 1]
        match_offset = None
        for shift in offsets:
            scratch = target / f'{row.dataset}.txt'
            try:
                count = normalize_raw_spmf(source, scratch, offset=shift)
                matched = (count == int(row.n_transactions) and
                           sha256_file(scratch) == str(row.clean_sha256).lower())
            except ValueError:
                matched = False
            if matched:
                match_offset = shift
                break
        if match_offset is None:
            raise AssertionError(f'Không tái tạo được hash: {row.dataset}')
        print('[PASS]', row.dataset, 'offset:', match_offset)
    print('Raw-to-normalized reproduction checked. Output kept separate from Drive.')
else:
    print('SKIP: raw-to-normalized reproduction (optional).')

## 8 · Tùy chọn: tái tạo Retail subsets — seed 2026

Thuật toán được giữ nhất quán với Notebook lịch sử: xáo trộn **chỉ số dòng** Retail một lần bằng `random.Random(2026).shuffle`, sau đó lấy prefix ở các tỷ lệ 10/25/50/75/100. Không được lấy trực tiếp các dòng đầu file gốc.

**Mặc định tắt.** File tái tạo ghi vào `/content/fim_reproduction_check/scalability`, đối chiếu hash với bản Drive; không ghi đè bản chuẩn.

In [ ]:
REGENERATE_RETAIL_SUBSETS = False

def regenerate_retail_subsets(source, target_dir, seed=2026):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    with Path(source).open('r', encoding='utf-8') as fi:
        transactions = [line.strip() for line in fi if line.strip()]
    assert len(transactions) == 88162, 'Retail transaction count changed'
    order = list(range(len(transactions)))
    random.Random(seed).shuffle(order)
    previous = set()
    reports = []
    for percentage in (10,25,50,75,100):
        n = len(transactions)*percentage//100
        chosen = order[:n]
        current = set(chosen)
        assert previous.issubset(current)
        previous = current
        file = target_dir / f'retail_p{percentage:03d}.txt'
        with file.open('w', encoding='utf-8', newline='\n') as fo:
            for index in chosen:
                fo.write(transactions[index] + '\n')
        reports.append({'dataset':file.name, 'transactions':n, 'sha256':sha256_file(file)})
    return pd.DataFrame(reports)

if REGENERATE_RETAIL_SUBSETS:
    recreated = regenerate_retail_subsets(DATA / 'retail.txt',
                       '/content/fim_reproduction_check/scalability')
    recreated['matches_existing'] = recreated.apply(
        lambda row: row['sha256'] == sha256_file(SCALABILITY / row['dataset']), axis=1)
    display(recreated)
    assert recreated['matches_existing'].all()
else:
    print('SKIP: Retail regeneration (optional).')

## 9 · Tùy chọn: tái tạo synthetic datasets — seed 2026

Giữ đúng cơ chế sinh dữ liệu đã dùng: **200 patterns gồm 4 item**, trong đó 125 pattern đầu bao phủ toàn bộ 500 item; số pattern được chọn theo trọng số; sinh **10.000 giao dịch nền dài 32 item**; tạo 6 mức độ dài bằng các prefix và hoán vị chiều dài. Dữ liệu theo chiều dài được xây dựng từ **cùng transaction nền**.

**Đây là dữ liệu tương quan (correlated), không phải ngẫu nhiên độc lập hoàn toàn.** Code tái tạo giữ nguyên thuật toán và random sequence của Notebook lịch sử. Không chạy nếu chưa cần audit.

In [ ]:
REGENERATE_SYNTHETIC = False

def regenerate_synthetic(target_dir, seed=2026):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    n_transactions, n_items, max_length = 10000, 500, 32
    targets = [5,10,15,20,25,30]
    pool_rng = random.Random(seed)
    item_ids = list(range(1, n_items+1))
    pool_rng.shuffle(item_ids)
    patterns = [item_ids[i:i+4] for i in range(0,n_items,4)]
    for _ in range(75):
        patterns.append(pool_rng.sample(range(1,n_items+1),4))
    assert len(patterns) == 200
    weights = [1 / (i+1)**0.8 for i in range(len(patterns))]
    rng = random.Random(seed+1)
    master = []
    for t in range(n_transactions):
        transaction, seen = [],set()
        n_patterns = rng.choices([1,2,3],weights=[25,50,25],k=1)[0]
        first_pattern = t if t<125 else rng.choices(range(200),weights=weights,k=1)[0]
        selected = [first_pattern]
        selected += rng.choices(range(200),weights=weights,k=n_patterns-1)
        for pattern_id in selected:
            for item in patterns[pattern_id]:
                if item not in seen:
                    seen.add(item)
                    transaction.append(item)
        while len(transaction)<max_length:
            item = rng.randint(1,n_items)
            if item not in seen:
                seen.add(item)
                transaction.append(item)
        master.append(transaction)
    statistics,previous_sets = [],None
    for length_target in targets:
        offsets = [-1,-1,0,1,1] if length_target==5 else [-2,-1,0,1,2]
        lengths = [length_target + offset for offset in offsets
                   for _ in range(n_transactions//len(offsets))]
        random.Random(seed+length_target).shuffle(lengths)
        filename = f'len{length_target:02d}.txt'
        destination = target_dir/filename
        current_sets,unique_items = [],set()
        with destination.open('w', encoding='utf-8',newline='\n') as fo:
            for i,length in enumerate(lengths):
                items=sorted(master[i][:length])
                assert len(items)==length and len(items)==len(set(items))
                current_sets.append(set(items))
                unique_items.update(items)
                fo.write(' '.join(map(str,items))+'\n')
        if previous_sets is not None:
            assert all(old.issubset(new) for old,new in zip(previous_sets,current_sets))
        previous_sets=current_sets
        avg_length=sum(lengths)/n_transactions
        assert avg_length==length_target and len(unique_items)==n_items
        statistics.append({'dataset': filename,'n_transactions':n_transactions,
                           'n_items':len(unique_items),'avg_length':avg_length,
                           'min_length':min(lengths),'max_length':max(lengths),
                           'density':avg_length/n_items,'seed':seed,
                           'sha256':sha256_file(destination)})
    summary=pd.DataFrame(statistics)
    summary.to_csv(target_dir/'synthetic_statistics.csv',index=False)
    return summary

if REGENERATE_SYNTHETIC:
    recreated = regenerate_synthetic('/content/fim_reproduction_check/synthetic')
    recreated['matches_existing'] = recreated.apply(
        lambda row: row['sha256'] == sha256_file(SYNTHETIC/row['dataset']),axis=1)
    display(recreated)
    assert recreated['matches_existing'].all()
else:
    print('SKIP: synthetic regeneration (optional).')

## 10 · Bàn giao và các giới hạn cần ghi trung thực

| Artifact | Mục đích | Vị trí |
|---|---|---|
| 9 `*.txt` | Dữ liệu chuẩn cho cả hai implementation | `data/benchmark/`; [Hugging Face](https://huggingface.co/datasets/hytrang/fpclose-fim-benchmarks) |
| `MANIFEST.md` | Provenance, normalization, hash, thống kê | `data/benchmark/MANIFEST.md` |
| `dataset_statistics.csv` | 9 dòng thống kê chi tiết | `results/dataset_statistics.csv` |
| `benchmark_inventory.csv` | 9 dataset và SHA-256 | `results/benchmark_inventory.csv` |
| 5 Retail subsets | Đầu vào thí nghiệm scalability | `data/scalability/` |
| 6 synthetic datasets | Đầu vào thí nghiệm transaction length | `data/synthetic/` |
| `02_Benchmark_FPClose` | Chọn lưới minsup 56 mức | Notebook thứ hai |

**Đã hoàn thành trước đây:** 9 datasets trên Hugging Face đã được tải lại và đối chiếu SHA-256 **9/9 PASS**.  
**Chưa được chứng minh ở bước chuẩn bị:** thuật toán của nhóm trùng SPMF tới từng itemset và support; runtime/RAM chính thức; so sánh base/opt; biểu đồ Chương 2.

**Lưu ý audit:** năm bản raw SPMF đầu có `id_offset = unknown_original_not_logged` trong hồ sơ lịch sử. Mọi chỉnh sửa provenance cần dựa trên bằng chứng tái tạo, không giả định. Thông tin link public trong MANIFEST cần được cập nhật nếu chưa có.

Không đưa các file dataset lớn vào GitHub: commit Notebook, script, MANIFEST, thống kê và CSV cấu hình; để dữ liệu lớn ở Hugging Face. Giữ notebook lịch sử làm bằng chứng quá trình thực hiện; notebook sạch là giao diện hướng dẫn tái lập/bàn giao.